In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT_DIR = Path(r"D:\Sami Data Set")
RAW_DIR = PROJECT_DIR / "data" / "raw"
DOCUMENTATION_DIR = PROJECT_DIR / "documentation"
TABLES_DIR = PROJECT_DIR / "outputs" / "tables"

DOCUMENTATION_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

parquet_files = sorted(RAW_DIR.glob("*.parquet"))

if not parquet_files:
    raise FileNotFoundError(f"No Parquet files found in: {RAW_DIR}")

print(f"Files available for audit: {len(parquet_files)}")

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT_DIR = Path(r"D:\Sami Data Set")
RAW_DIR = PROJECT_DIR / "data" / "raw"
DOCUMENTATION_DIR = PROJECT_DIR / "documentation"
TABLES_DIR = PROJECT_DIR / "outputs" / "tables"

DOCUMENTATION_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

parquet_files = sorted(RAW_DIR.glob("*.parquet"))

if not parquet_files:
    raise FileNotFoundError(
        f"No Parquet files were found in:\n{RAW_DIR}"
    )

print(f"Files available for audit: {len(parquet_files)}")

Files available for audit: 10


In [5]:
schema_records = []
reference_columns = None

for file_path in parquet_files:
    parquet_file = pq.ParquetFile(file_path)
    columns = parquet_file.schema.names

    if reference_columns is None:
        reference_columns = columns

    schema_records.append({
        "file_name": file_path.name,
        "column_count": len(columns),
        "matches_reference_schema": columns == reference_columns
    })

schema_summary = pd.DataFrame(schema_records)

display(schema_summary)

schema_summary.to_csv(
    DOCUMENTATION_DIR / "schema_consistency.csv",
    index=False
)

print("All schemas match:",
      schema_summary["matches_reference_schema"].all())

,file_name,column_count,matches_reference_schema
0,Botnet-Friday-02-03-2018_TrafficForML_CICFlowM...,78,True
1,Bruteforce-Wednesday-14-02-2018_TrafficForML_C...,78,True
2,DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowM...,78,True
3,DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlo...,78,True
4,DoS1-Thursday-15-02-2018_TrafficForML_CICFlowM...,78,True
5,DoS2-Friday-16-02-2018_TrafficForML_CICFlowMet...,78,True
6,Infil1-Wednesday-28-02-2018_TrafficForML_CICFl...,78,True
7,Infil2-Thursday-01-03-2018_TrafficForML_CICFlo...,78,True
8,Web1-Thursday-22-02-2018_TrafficForML_CICFlowM...,78,True
9,Web2-Friday-23-02-2018_TrafficForML_CICFlowMet...,78,True


All schemas match: True


In [7]:
# Inspect schema and a small sample safely

sample_file = parquet_files[0]
parquet_sample = pq.ParquetFile(sample_file)

# Read only five records
first_batch = next(parquet_sample.iter_batches(batch_size=5))
sample_df = first_batch.to_pandas()

print("Sample file:")
print(sample_file.name)

print("\nNumber of columns:")
print(len(parquet_sample.schema.names))

print("\nColumns and data types:")
for number, field in enumerate(parquet_sample.schema_arrow, start=1):
    print(f"{number:02d}. {field.name} — {field.type}")

print("\nFirst five records:")
display(sample_df.head())

Sample file:
Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet

Number of columns:
78

Columns and data types:
01. Protocol — int8
02. Flow Duration — int32
03. Total Fwd Packets — int32
04. Total Backward Packets — int32
05. Fwd Packets Length Total — int32
06. Bwd Packets Length Total — float
07. Fwd Packet Length Max — int16
08. Fwd Packet Length Min — int16
09. Fwd Packet Length Mean — float
10. Fwd Packet Length Std — float
11. Bwd Packet Length Max — int16
12. Bwd Packet Length Min — int16
13. Bwd Packet Length Mean — float
14. Bwd Packet Length Std — float
15. Flow Bytes/s — double
16. Flow Packets/s — double
17. Flow IAT Mean — float
18. Flow IAT Std — float
19. Flow IAT Max — float
20. Flow IAT Min — float
21. Fwd IAT Total — float
22. Fwd IAT Mean — float
23. Fwd IAT Std — float
24. Fwd IAT Max — float
25. Fwd IAT Min — float
26. Bwd IAT Total — float
27. Bwd IAT Mean — float
28. Bwd IAT Std — float
29. Bwd IAT Max — float
30. Bwd IAT Min — float
31. Fwd PSH Flags — 

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,141385,9,7,553,3773.0,202,0,61.444443,87.534439,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1,6,281,2,1,38,0.0,38,0,19.000000,26.870058,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
2,6,279824,11,15,1086,10527.0,385,0,98.727272,129.392502,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
3,6,132,2,0,0,0.0,0,0,0.000000,0.000000,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
4,6,274016,9,13,1285,6141.0,517,0,142.777771,183.887726,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign


In [8]:
# Create an initial data dictionary from the Parquet schema

schema = pq.ParquetFile(parquet_files[0]).schema_arrow

column_records = []

for position, field in enumerate(schema, start=1):
    column_records.append({
        "position": position,
        "column_name": field.name,
        "parquet_type": str(field.type),
        "is_label": field.name.strip().lower() == "label"
    })

column_dictionary = pd.DataFrame(column_records)

display(column_dictionary)

column_dictionary.to_csv(
    DOCUMENTATION_DIR / "initial_data_dictionary.csv",
    index=False
)

print("Data dictionary saved successfully.")

,position,column_name,parquet_type,is_label
0,1,Protocol,int8,False
1,2,Flow Duration,int32,False
2,3,Total Fwd Packets,int32,False
3,4,Total Backward Packets,int32,False
4,5,Fwd Packets Length Total,int32,False
...,...,...,...,...
73,74,Idle Mean,float,False
74,75,Idle Std,float,False
75,76,Idle Max,float,False
76,77,Idle Min,float,False


Data dictionary saved successfully.


In [9]:
review_keywords = [
    "label",
    "class",
    "attack",
    "flow id",
    "source ip",
    "destination ip",
    "src ip",
    "dst ip",
    "timestamp"
]

columns_for_review = column_dictionary[
    column_dictionary["column_name"]
    .str.lower()
    .apply(
        lambda name: any(
            keyword in name for keyword in review_keywords
        )
    )
]

display(columns_for_review)

,position,column_name,parquet_type,is_label
77,78,Label,"dictionary<values=string, indices=int8, ordere...",True


In [10]:
label_records = []

for file_number, file_path in enumerate(parquet_files, start=1):
    print(
        f"[{file_number}/{len(parquet_files)}] "
        f"Reading labels from {file_path.name}"
    )

    label_table = pq.read_table(
        file_path,
        columns=["Label"]
    )

    labels = (
        label_table
        .column("Label")
        .to_pandas()
        .astype("string")
        .str.strip()
    )

    counts = labels.value_counts(dropna=False)

    for label, count in counts.items():
        label_records.append({
            "file_name": file_path.name,
            "original_attack_label": str(label),
            "count": int(count)
        })

label_counts_by_file = pd.DataFrame(label_records)

combined_label_distribution = (
    label_counts_by_file
    .groupby("original_attack_label", as_index=False)["count"]
    .sum()
    .sort_values("count", ascending=False)
)

combined_label_distribution["percentage"] = (
    combined_label_distribution["count"]
    / combined_label_distribution["count"].sum()
    * 100
).round(4)

display(combined_label_distribution)

label_counts_by_file.to_csv(
    TABLES_DIR / "label_counts_by_file.csv",
    index=False
)

combined_label_distribution.to_csv(
    TABLES_DIR / "combined_attack_label_distribution.csv",
    index=False
)

print(
    "\nTotal labelled records:",
    combined_label_distribution["count"].sum()
)

[1/10] Reading labels from Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet
[2/10] Reading labels from Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet
[3/10] Reading labels from DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet
[4/10] Reading labels from DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet
[5/10] Reading labels from DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter.parquet
[6/10] Reading labels from DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter.parquet
[7/10] Reading labels from Infil1-Wednesday-28-02-2018_TrafficForML_CICFlowMeter.parquet
[8/10] Reading labels from Infil2-Thursday-01-03-2018_TrafficForML_CICFlowMeter.parquet
[9/10] Reading labels from Web1-Thursday-22-02-2018_TrafficForML_CICFlowMeter.parquet
[10/10] Reading labels from Web2-Friday-23-02-2018_TrafficForML_CICFlowMeter.parquet


,original_attack_label,count,percentage
0,Benign,5329008,80.0208
6,DDoS attacks-LOIC-HTTP,575364,8.6397
4,DDOS attack-HOIC,198861,2.9861
8,DoS attacks-Hulk,145199,2.1803
1,Bot,144535,2.1703
12,Infilteration,118483,1.7791
14,SSH-Bruteforce,94048,1.4122
7,DoS attacks-GoldenEye,41406,0.6218
10,DoS attacks-Slowloris,9908,0.1488
5,DDOS attack-LOIC-UDP,1730,0.0260



Total labelled records: 6659532


In [11]:
binary_distribution = combined_label_distribution.copy()

binary_distribution["binary_label"] = np.where(
    binary_distribution["original_attack_label"]
    .str.lower()
    .eq("benign"),
    0,
    1
)

binary_distribution = (
    binary_distribution
    .groupby("binary_label", as_index=False)["count"]
    .sum()
)

binary_distribution["traffic_type"] = (
    binary_distribution["binary_label"]
    .map({
        0: "Benign",
        1: "Malicious"
    })
)

binary_distribution["percentage"] = (
    binary_distribution["count"]
    / binary_distribution["count"].sum()
    * 100
).round(4)

binary_distribution = binary_distribution[
    [
        "binary_label",
        "traffic_type",
        "count",
        "percentage"
    ]
]

display(binary_distribution)

binary_distribution.to_csv(
    TABLES_DIR / "binary_class_distribution.csv",
    index=False
)

,binary_label,traffic_type,count,percentage
0,0,Benign,5329008,80.0208
1,1,Malicious,1330524,19.9792


In [ ]:
from pathlib import Path
import gc
import time

import numpy as np
import pandas as pd

HASH_DIR = TABLES_DIR / "row_hashes"
HASH_DIR.mkdir(parents=True, exist_ok=True)

audit_records = []
missing_records = []
infinite_records = []

for file_number, file_path in enumerate(parquet_files, start=1):
    start_time = time.time()

    print(
        f"[{file_number}/{len(parquet_files)}] "
        f"Auditing: {file_path.name}"
    )

    try:
        # Load only one file at a time
        df = pd.read_parquet(file_path)
        df.columns = df.columns.astype(str).str.strip()

        if "Label" not in df.columns:
            raise KeyError("Label column is missing.")

        # Standardise the label text for reliable comparison
        df["Label"] = df["Label"].astype("string").str.strip()

        # Missing-value audit
        missing_counts = df.isna().sum()

        for column, count in missing_counts[missing_counts > 0].items():
            missing_records.append({
                "file_name": file_path.name,
                "column": column,
                "missing_count": int(count)
            })

        # Infinity audit
        numeric_columns = df.select_dtypes(
            include=[np.number]
        ).columns

        file_positive_infinity = 0
        file_negative_infinity = 0

        for column in numeric_columns:
            values = df[column].to_numpy(copy=False)

            # Infinity can only occur in floating-point columns
            if np.issubdtype(values.dtype, np.floating):
                positive_count = int(np.isposinf(values).sum())
                negative_count = int(np.isneginf(values).sum())

                if positive_count > 0 or negative_count > 0:
                    infinite_records.append({
                        "file_name": file_path.name,
                        "column": column,
                        "positive_infinity": positive_count,
                        "negative_infinity": negative_count,
                        "total_infinity": (
                            positive_count + negative_count
                        )
                    })

                file_positive_infinity += positive_count
                file_negative_infinity += negative_count

        # Exact duplicates within the current file
        duplicate_count = int(df.duplicated().sum())

        # Save row hashes for later cross-file duplicate checking
        row_hashes = pd.util.hash_pandas_object(
            df,
            index=False
        ).to_numpy(dtype=np.uint64)

        np.save(
            HASH_DIR / f"{file_path.stem}_hashes.npy",
            row_hashes
        )

        audit_records.append({
            "file_name": file_path.name,
            "rows": int(len(df)),
            "columns": int(len(df.columns)),
            "missing_values": int(missing_counts.sum()),
            "positive_infinity": file_positive_infinity,
            "negative_infinity": file_negative_infinity,
            "total_infinity": (
                file_positive_infinity +
                file_negative_infinity
            ),
            "duplicates_within_file": duplicate_count,
            "unique_labels": int(df["Label"].nunique()),
            "processing_seconds": round(
                time.time() - start_time,
                2
            ),
            "status": "Completed",
            "error": ""
        })

        print(
            f"Completed — rows: {len(df):,}, "
            f"missing: {int(missing_counts.sum()):,}, "
            f"infinity: "
            f"{file_positive_infinity + file_negative_infinity:,}, "
            f"duplicates: {duplicate_count:,}"
        )

        del df
        del row_hashes
        gc.collect()

    except Exception as error:
        audit_records.append({
            "file_name": file_path.name,
            "rows": None,
            "columns": None,
            "missing_values": None,
            "positive_infinity": None,
            "negative_infinity": None,
            "total_infinity": None,
            "duplicates_within_file": None,
            "unique_labels": None,
            "processing_seconds": round(
                time.time() - start_time,
                2
            ),
            "status": "Error",
            "error": str(error)
        })

        print("Error:", error)

audit_summary = pd.DataFrame(audit_records)

missing_summary = pd.DataFrame(
    missing_records,
    columns=["file_name", "column", "missing_count"]
)

infinite_summary = pd.DataFrame(
    infinite_records,
    columns=[
        "file_name",
        "column",
        "positive_infinity",
        "negative_infinity",
        "total_infinity"
    ]
)

display(audit_summary)

In [ ]:
audit_summary.to_csv(
    DOCUMENTATION_DIR / "data_quality_audit_by_file.csv",
    index=False
)

missing_summary.to_csv(
    TABLES_DIR / "missing_values_by_file.csv",
    index=False
)

infinite_summary.to_csv(
    TABLES_DIR / "infinite_values_by_file.csv",
    index=False
)

print("Quality-audit files saved successfully.")

print("\nTotal missing values:")
print(audit_summary["missing_values"].sum())

print("\nTotal infinity values:")
print(audit_summary["total_infinity"].sum())

print("\nDuplicates found within individual files:")
print(audit_summary["duplicates_within_file"].sum())

In [ ]:
hash_files = sorted(HASH_DIR.glob("*_hashes.npy"))

if len(hash_files) != len(parquet_files):
    raise RuntimeError(
        "Not all row-hash files were generated. "
        "Complete the quality audit first."
    )

hash_arrays = [
    np.load(hash_file, mmap_mode="r")
    for hash_file in hash_files
]

total_hashes = sum(len(array) for array in hash_arrays)

print(f"Total row hashes: {total_hashes:,}")

all_hashes = np.empty(total_hashes, dtype=np.uint64)

start = 0

for array in hash_arrays:
    end = start + len(array)
    all_hashes[start:end] = array
    start = end

unique_hashes, hash_counts = np.unique(
    all_hashes,
    return_counts=True
)

duplicate_rows_across_dataset = int(
    np.maximum(hash_counts - 1, 0).sum()
)

duplicate_groups = int((hash_counts > 1).sum())

global_duplicate_summary = pd.DataFrame([{
    "total_rows": total_hashes,
    "unique_row_hashes": len(unique_hashes),
    "duplicate_groups": duplicate_groups,
    "duplicate_rows_beyond_first_occurrence":
        duplicate_rows_across_dataset
}])

display(global_duplicate_summary)

global_duplicate_summary.to_csv(
    TABLES_DIR / "global_duplicate_summary.csv",
    index=False
)

del all_hashes
del unique_hashes
del hash_counts
gc.collect()

In [ ]:
print("Jupyter is Working")

In [ ]:
print("Jupyter is working")

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT_DIR = Path(r"D:\Sami Data Set")
RAW_DIR = PROJECT_DIR / "data" / "raw"
DOCUMENTATION_DIR = PROJECT_DIR / "documentation"
TABLES_DIR = PROJECT_DIR / "outputs" / "tables"

DOCUMENTATION_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

parquet_files = sorted(RAW_DIR.glob("*.parquet"))

print("Files found:", len(parquet_files))

if parquet_files:
    print("First file:", parquet_files[0].name)

Files found: 10
First file: Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet


In [3]:
test_file = parquet_files[0]

print("Reading:", test_file.name)

test_df = pd.read_parquet(test_file)

print("Rows:", f"{len(test_df):,}")
print("Columns:", len(test_df.columns))
print("Missing values:", int(test_df.isna().sum().sum()))
print("Unique labels:", test_df["Label"].nunique())

display(
    test_df["Label"]
    .astype("string")
    .str.strip()
    .value_counts()
    .rename_axis("Label")
    .reset_index(name="Count")
)


Reading: Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet
Rows: 771,587
Columns: 78
Missing values: 0
Unique labels: 2


,Label,Count
0,Benign,627052
1,Bot,144535


In [4]:
import gc

del test_df
gc.collect()

print("Test data removed from memory.")

Test data removed from memory.


In [5]:
import gc
import time

import numpy as np
import pandas as pd

audit_records = []
missing_records = []
infinite_records = []
label_records = []

for file_number, file_path in enumerate(parquet_files, start=1):
    start_time = time.time()

    print(
        f"\n[{file_number}/{len(parquet_files)}] "
        f"Reading {file_path.name}",
        flush=True
    )

    try:
        df = pd.read_parquet(file_path)
        df.columns = df.columns.astype(str).str.strip()

        if "Label" not in df.columns:
            raise KeyError("Label column not found.")

        df["Label"] = df["Label"].astype("string").str.strip()

        # Missing values
        missing_counts = df.isna().sum()
        total_missing = int(missing_counts.sum())

        for column, count in missing_counts[missing_counts > 0].items():
            missing_records.append({
                "file_name": file_path.name,
                "column": column,
                "missing_count": int(count)
            })

        # Positive and negative infinity
        positive_infinity = 0
        negative_infinity = 0

        numeric_columns = df.select_dtypes(include=[np.number]).columns

        for column in numeric_columns:
            values = df[column].to_numpy(copy=False)

            if np.issubdtype(values.dtype, np.floating):
                positive_count = int(np.isposinf(values).sum())
                negative_count = int(np.isneginf(values).sum())

                positive_infinity += positive_count
                negative_infinity += negative_count

                if positive_count > 0 or negative_count > 0:
                    infinite_records.append({
                        "file_name": file_path.name,
                        "column": column,
                        "positive_infinity": positive_count,
                        "negative_infinity": negative_count,
                        "total_infinity": positive_count + negative_count
                    })

        # Exact duplicates within this file
        duplicates = int(df.duplicated().sum())

        # Label counts
        label_counts = df["Label"].value_counts(dropna=False)

        for label, count in label_counts.items():
            label_records.append({
                "file_name": file_path.name,
                "original_attack_label": str(label),
                "count": int(count)
            })

        audit_records.append({
            "file_name": file_path.name,
            "rows": int(len(df)),
            "columns": int(len(df.columns)),
            "missing_values": total_missing,
            "positive_infinity": positive_infinity,
            "negative_infinity": negative_infinity,
            "total_infinity": positive_infinity + negative_infinity,
            "duplicates_within_file": duplicates,
            "unique_labels": int(df["Label"].nunique()),
            "processing_seconds": round(time.time() - start_time, 2),
            "status": "Completed",
            "error": ""
        })

        print(
            f"Completed | Rows: {len(df):,} | "
            f"Missing: {total_missing:,} | "
            f"Infinity: {positive_infinity + negative_infinity:,} | "
            f"Duplicates: {duplicates:,}",
            flush=True
        )

        del df
        gc.collect()

    except Exception as error:
        audit_records.append({
            "file_name": file_path.name,
            "rows": None,
            "columns": None,
            "missing_values": None,
            "positive_infinity": None,
            "negative_infinity": None,
            "total_infinity": None,
            "duplicates_within_file": None,
            "unique_labels": None,
            "processing_seconds": round(time.time() - start_time, 2),
            "status": "Error",
            "error": str(error)
        })

        print(f"Error: {error}", flush=True)

audit_summary = pd.DataFrame(audit_records)

missing_summary = pd.DataFrame(
    missing_records,
    columns=["file_name", "column", "missing_count"]
)

infinite_summary = pd.DataFrame(
    infinite_records,
    columns=[
        "file_name",
        "column",
        "positive_infinity",
        "negative_infinity",
        "total_infinity"
    ]
)

label_counts_by_file = pd.DataFrame(label_records)

display(audit_summary)


[1/10] Reading Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet
Completed | Rows: 771,587 | Missing: 0 | Infinity: 0 | Duplicates: 0

[2/10] Reading Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet
Completed | Rows: 619,346 | Missing: 0 | Infinity: 0 | Duplicates: 0

[3/10] Reading DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet
Completed | Rows: 954,846 | Missing: 0 | Infinity: 0 | Duplicates: 0

[4/10] Reading DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet
Completed | Rows: 561,396 | Missing: 0 | Infinity: 0 | Duplicates: 0

[5/10] Reading DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter.parquet
Completed | Rows: 794,812 | Missing: 0 | Infinity: 0 | Duplicates: 0

[6/10] Reading DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter.parquet
Completed | Rows: 591,873 | Missing: 0 | Infinity: 0 | Duplicates: 0

[7/10] Reading Infil1-Wednesday-28-02-2018_TrafficForML_CICFlowMeter.parquet
Completed | Rows: 456,873 | Missing: 0 | Infinity: 

,file_name,rows,columns,missing_values,positive_infinity,negative_infinity,total_infinity,duplicates_within_file,unique_labels,processing_seconds,status,error
0,Botnet-Friday-02-03-2018_TrafficForML_CICFlowM...,771587,78,0,0,0,0,0,2,8.70,Completed,
1,Bruteforce-Wednesday-14-02-2018_TrafficForML_C...,619346,78,0,0,0,0,0,3,8.32,Completed,
2,DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowM...,954846,78,0,0,0,0,0,2,10.81,Completed,
3,DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlo...,561396,78,0,0,0,0,0,3,4.13,Completed,
4,DoS1-Thursday-15-02-2018_TrafficForML_CICFlowM...,794812,78,0,0,0,0,0,3,7.85,Completed,
5,DoS2-Friday-16-02-2018_TrafficForML_CICFlowMet...,591873,78,0,0,0,0,0,3,4.93,Completed,
6,Infil1-Wednesday-28-02-2018_TrafficForML_CICFl...,456873,78,0,0,0,0,0,2,4.29,Completed,
7,Infil2-Thursday-01-03-2018_TrafficForML_CICFlo...,249170,78,0,0,0,0,0,2,2.19,Completed,
8,Web1-Thursday-22-02-2018_TrafficForML_CICFlowM...,830224,78,0,0,0,0,0,4,8.65,Completed,
9,Web2-Friday-23-02-2018_TrafficForML_CICFlowMet...,829405,78,0,0,0,0,0,4,8.39,Completed,


In [6]:
audit_summary.to_csv(
    DOCUMENTATION_DIR / "data_quality_audit_by_file.csv",
    index=False
)

missing_summary.to_csv(
    TABLES_DIR / "missing_values_by_file.csv",
    index=False
)

infinite_summary.to_csv(
    TABLES_DIR / "infinite_values_by_file.csv",
    index=False
)

label_counts_by_file.to_csv(
    TABLES_DIR / "label_counts_by_file.csv",
    index=False
)

print("Audit files saved successfully.")

print("\nTotal audited rows:")
print(f"{audit_summary['rows'].sum():,.0f}")

print("\nTotal missing values:")
print(f"{audit_summary['missing_values'].sum():,.0f}")

print("\nTotal infinity values:")
print(f"{audit_summary['total_infinity'].sum():,.0f}")

print("\nDuplicates within files:")
print(f"{audit_summary['duplicates_within_file'].sum():,.0f}")

Audit files saved successfully.

Total audited rows:
6,659,532

Total missing values:
0

Total infinity values:
0

Duplicates within files:
0


## Data-Quality Audit Conclusion

The audit covered 10 Parquet files containing 6,659,532 network-flow
records and 78 consistently structured columns. No missing values,
positive or negative infinite values, or exact duplicate records within
individual files were identified.

The downloaded feature-level version therefore required no imputation
or removal of within-file duplicates. Identifier fields such as source
IP address, destination IP address, Flow ID and timestamp were not
present in the supplied files. These fields were therefore not removed
by the project team.

The original files remain unchanged in the raw-data directory. Further
processing will add the original attack label and binary classification
label while preserving a documented record of all decisions.
    